# Tool Use & Function Calling — a step-by-step, runnable teaching notebook

This notebook builds a **real function-calling agent** from the ground up, one operation at a time, driving a genuine small instruction-tuned LLM with *native* tool calling (`Qwen/Qwen2.5-1.5B-Instruct`) through the real **structured** protocol: JSON-schema tool declarations → the model emits a structured `<tool_call>{...}</tool_call>` → we parse the JSON, validate the arguments against the schema, run the real tool, feed the result back as a **tool-role message**, and let the model answer. It is the executable companion to the chapter and to `function_calling_agent.py` — every function used here lives in that module, imported so the notebook and the module can never drift apart.

Nothing about the model's output is mocked. The traces below are what the model actually generates (greedy / temperature 0, so they are reproducible). By the end you will have **seen**, with a real model and real tools:

1. why free-text tool invocation (the ReAct way) is **brittle** — and a real measurement of it;
2. three **real JSON-schema tools** — a safe calculator, a unit converter, an FX-rate lookup;
3. how the schemas enter the prompt via `apply_chat_template(tools=...)`;
4. a **real structured `<tool_call>`** the model emits, parsed as JSON (not regex-on-prose);
5. **schema validation** — because valid JSON is not the same as valid arguments;
6. the **full protocol loop** — call → tool-role result → answer — on single, sequential, and **parallel** multi-tool tasks;
7. a **head-to-head**: structured function-calling vs a ReAct-style text protocol, on reliability.

The first run downloads the model (a few hundred MB) and caches it; every run after is offline and reproducible. It runs on CPU, Apple MPS, or CUDA — whatever you have.

## Step 0 — Setup and version banner

We import the real pieces from the chapter module (so this notebook uses the *exact same code* the chapter and figures use) and print the library + model versions and the device the results were produced on. `pick_device()` chooses `cuda → mps → cpu` without assuming a GPU.

In [1]:
import torch
import transformers

from function_calling_agent import pick_device

print(f'torch {torch.__version__} | transformers {transformers.__version__} '
      f'| device {pick_device()}')

torch 2.12.0 | transformers 5.10.2 | device mps


## Step 1 — The problem: free-text tool calls are brittle

The sibling **ReAct** chapter has the model write its tool call as *free text* — `Action: calculator[481 * 32 + 19]` — which the runtime then **regex-parses**. That works until the model drifts: a missing bracket, prose instead of the line, the wrong shape. Feel that brittleness directly. We give the model a ReAct-style text instruction and a query, and try to parse a call out of whatever prose it returns.

In [2]:
from function_calling_agent import ToolCallingModel, _TEXT_PROTOCOL_SYSTEM, _parse_text_call

model = ToolCallingModel()   # loads the real model (downloads+caches on first run)
print(f'loaded {model.model_id} on {model.device}\n')

q = 'What is 481 multiplied by 32, then plus 19?'
text_raw = model.generate([
    {'role': 'system', 'content': _TEXT_PROTOCOL_SYSTEM},
    {'role': 'user', 'content': q},
])
print('--- the model prose ---')
print(text_raw[:300])
print('\n--- regex-parsed call ---')
print(_parse_text_call(text_raw))   # often None: the prose did not match the TOOL: grammar

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-1.5B-Instruct on mps



--- the model prose ---
calculator(481 * 32 + 19)

--- regex-parsed call ---
None


The model answers helpfully but *not* in the rigid `TOOL: name(args)` shape the parser needs — so the parse returns `None`. This is the felt inadequacy function calling removes: instead of hoping the model formats prose correctly, we **declare a schema** and rely on the model having been trained to emit a **structured** call the runtime can parse as data.

## Step 2 — Real tool #1: a safe calculator (AST-walked, not `eval`)

The first tool is a real calculator. Crucially it is **not** `eval(expr)` — the model's arguments are untrusted input, exactly like user input. Instead we parse the expression to an abstract syntax tree and walk it, permitting *only* numbers and a whitelist of arithmetic operators. It also cleans the model's frequent `^` (caret) into Python's `**`. Safe tool design is part of the lesson.

In [3]:
from function_calling_agent import calculator

print(calculator(expression='481 * 32 + 19'))   # the real answer
print(calculator(expression='17^3 - 200'))      # caret power is normalised to ** -> 4713
print(calculator(expression='(1287 - 998) * 6'))# parentheses respected

# and it refuses anything that is not pure arithmetic (this is what makes it SAFE):
try:
    calculator(expression='__import__("os").system("echo hi")')
except Exception as e:
    print('rejected unsafe input:', type(e).__name__, '-', e)

15411
4713
1734
rejected unsafe input: ToolError - expression contains a disallowed operation


## Step 3 — Real tools #2 and #3: a unit converter and an FX-rate lookup

Two more real tools give the agent a reason to *choose* between tools and to *chain* them. `convert_units` converts length/mass/temperature over a genuine factor table (with unit-alias normalisation and a real error path); `get_exchange_rate` returns a real (offline) rate the agent must then *use*. Both are ordinary Python functions — the model never runs them, our runtime does.

In [4]:
from function_calling_agent import convert_units, get_exchange_rate

print(convert_units(value=42, from_unit='km', to_unit='mi'))       # 26.0976 mi
print(convert_units(value=100, from_unit='Celsius', to_unit='F'))  # alias 'Celsius' -> C; 212 F
print(get_exchange_rate(from_currency='USD', to_currency='JPY'))   # a real rate: 157.0

# real error paths (the schema guarantees args ARRIVE, not that they name real, compatible units):
try:
    convert_units(value=5, from_unit='kg', to_unit='mi')  # mass -> length is incompatible
except Exception as e:
    print('rejected:', e)

26.0976 mi
212 F
157.0
rejected: incompatible dimensions: mass (kg) vs length (mi)


## Step 4 — The JSON schema: what the model actually sees

This is the pivot from ReAct. Each tool is declared with a **JSON schema** — a name, a description, and *typed* parameters with a `required` list. The registry pairs each schema with its real callable, so the declaration handed to the model and the function the runtime runs can never drift. Here is the real schema for the calculator.

In [5]:
import json

from function_calling_agent import TOOL_REGISTRY, tool_schemas

print('registered tools:', list(TOOL_REGISTRY))
print()
print(json.dumps(TOOL_REGISTRY['calculator'].schema, indent=2))

registered tools: ['calculator', 'convert_units', 'get_exchange_rate']

{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression and return the numeric result. Use for any exact calculation (multiplication, powers, parenthesised expressions).",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "An arithmetic expression, e.g. '481 * 32 + 19'."
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}


## Step 5 — The schemas enter the prompt via `apply_chat_template(tools=...)`

How does the model *know* these tools exist? The chat template has a `tools=` slot. When we render the messages with `apply_chat_template(messages, tools=tool_schemas(), ...)`, the tokenizer injects the JSON tool declarations into the exact system section the model was **trained** to read. Let's look at the rendered prompt so the mechanism is not magic.

In [6]:
rendered = model.tokenizer.apply_chat_template(
    [{'role': 'user', 'content': q}],
    tools=tool_schemas(),
    add_generation_prompt=True,
    tokenize=False,
)
# show the part that declares the tools (the template wraps them in a <tools>...</tools> block)
start = rendered.find('You are')
print(rendered[start:start + 900])

You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic expression and return the numeric result. Use for any exact calculation (multiplication, powers, parenthesised expressions).", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "An arithmetic expression, e.g. '481 * 32 + 19'."}}, "required": ["expression"]}}}
{"type": "function", "function": {"name": "convert_units", "description": "Convert a value from one unit to another. Supports length (km, m, mi, ft), mass (kg, g, lb), and temperature (C, F).", "parameters": {"type": "object", "properties": {"value": {"type": "number", 


## Step 6 — A real structured `<tool_call>`: parse JSON, not prose

Now the payoff. We generate *with* the tool schemas and the model emits a **structured** tool call wrapped in `<tool_call>...</tool_call>` — valid JSON with a `name` and an `arguments` object. `parse_tool_calls` pulls out the JSON and `json.loads` it. Compare this to Step 1: we are parsing a **data format** with an unambiguous grammar, not guessing at free text.

In [7]:
from function_calling_agent import parse_tool_calls

raw = model.generate([{'role': 'user', 'content': q}], tools=tool_schemas())
print('--- raw model output ---')
print(raw)
print('\n--- parsed structured calls ---')
calls = parse_tool_calls(raw)
for c in calls:
    print(c)

--- raw model output ---
<tool_call>
{"name": "calculator", "arguments": {"expression": "481 * 32 + 19"}}
</tool_call>

--- parsed structured calls ---
ToolCall(name='calculator', arguments={'expression': '481 * 32 + 19'})


## Step 7 — Validation: valid JSON is not the same as valid arguments

Structured calling guarantees the *shape* of the arguments — never their *correctness*. The model can omit a required field, send the wrong type, or (famously) drop parentheses so `(1287 - 998) * 6` becomes `1287 - 998 * 6`. `validate_arguments` checks every call against its schema (required keys present, types coercible) and raises a real error on a violation — which the loop turns into a tool-result the model can read and fix.

In [8]:
from function_calling_agent import ToolCall, validate_arguments

good = ToolCall(name='calculator', arguments={'expression': '481 * 32 + 19'})
print('valid   ->', validate_arguments(good, TOOL_REGISTRY['calculator']))

missing = ToolCall(name='convert_units', arguments={'value': 42, 'from_unit': 'km'})  # no to_unit
try:
    validate_arguments(missing, TOOL_REGISTRY['convert_units'])
except Exception as e:
    print('invalid ->', e)

valid   -> {'expression': '481 * 32 + 19'}
invalid -> missing required argument(s) ['to_unit'] for convert_units


## Step 8 — Dispatch: validate, then run the real tool

`dispatch` ties Steps 2–7 together: look the tool up in the registry, validate the arguments against its schema, call the real Python function, and return its real result string. Unknown tools and validation/tool errors become *result strings* (not crashes), so the agent can read the problem and recover — how a robust function-calling loop behaves in the wild.

In [9]:
from function_calling_agent import dispatch

print('good call   ->', dispatch(calls[0]))
print('bad tool    ->', dispatch(ToolCall(name='translate', arguments={'text': 'hi'})))
print('bad args    ->', dispatch(ToolCall(name='convert_units', arguments={'value': 5,
                                          'from_unit': 'kg', 'to_unit': 'mi'})))

good call   -> 15411
bad tool    -> Error: no tool named 'translate'. Available: calculator, convert_units, get_exchange_rate.
bad args    -> Error: incompatible dimensions: mass (kg) vs length (mi)


## Step 9 — The full protocol loop on a single-tool task

Now assemble everything into the loop. `run_agent` builds the message list by the book: the user query, then for each turn an *assistant* message recording the structured `tool_calls`, then one **tool-role** result message per call, then the model is asked again — until it answers with no tool call (that reply is the answer) or the turn budget is hit. Here is the complete real trace for the query the text parser choked on in Step 1.

In [10]:
from function_calling_agent import run_agent

result = run_agent(model, q)
print(result.transcript())
print()
print(f'stop_reason={result.stop_reason} | turns={result.num_turns} | '
      f'tool_calls={result.num_tool_calls} | answer={result.answer!r}')

User: What is 481 multiplied by 32, then plus 19?
Assistant -> tool_call: calculator({"expression": "481 * 32 + 19"})
Tool [calculator] -> 15411
Assistant: The result of multiplying 481 by 32 and then adding 19 is 15,411.

stop_reason=answered | turns=1 | tool_calls=1 | answer='The result of multiplying 481 by 32 and then adding 19 is 15,411.'


The model emitted a structured call, our runtime executed the real calculator, we handed the result back as a tool-role message, and the model produced a grounded final answer. That round-trip — **schema → call → execute → tool-result → answer** — is the entire function-calling protocol.

## Step 10 — A sequential multi-tool trace: look up, THEN compute

The power shows on tasks that need one tool's result to drive the next. The agent looks up the exchange rate, **reads the real rate**, then grounds its answer on it. It has to decide *which* tool to call first based on the task, and *what to do* based on what the first tool returned.

In [11]:
seq = run_agent(model, 'First look up the USD to JPY exchange rate with the get_exchange_rate tool, '
                       'then multiply 40 dollars by that rate to get the yen amount.')
print(seq.transcript())

User: First look up the USD to JPY exchange rate with the get_exchange_rate tool, then multiply 40 dollars by that rate to get the yen amount.
Assistant -> tool_call: get_exchange_rate({"from_currency": "USD", "to_currency": "JPY"})
Tool [get_exchange_rate] -> 157.0
Assistant: Now, let's use the calculated exchange rate of 157.0 to multiply it by 40 dollars to find out how many Japanese Yen you would have:

40 * 157 = 6,280 Yen

Therefore, 40 US Dollars is approximately equal to 6,280 Japanese Yen.


The rate **157.0** came from the tool, not from the model's memory — and the final answer is built on that real observation. (Notice the model does the final multiply in prose here; a common, realistic behaviour. The lesson stands: the *fact* was grounded in a real tool result.)

## Step 11 — Parallel tool calls: two independent calls in ONE turn

When two sub-tasks are *independent*, a capable model emits **both** tool calls in a single turn — and because `parse_tool_calls` returns a *list*, our loop dispatches both and returns a tool-result for each. This is a real efficiency win (no need to round-trip twice) and it is why structured calling models the tool_calls as a list, not a single call.

In [12]:
par = run_agent(model, 'Convert 42 kilometres to miles, and separately convert 5 kilograms to pounds.')
print(par.transcript())
print()
print(f'tool calls this run: {par.num_tool_calls} across {par.num_turns} turn(s)')

User: Convert 42 kilometres to miles, and separately convert 5 kilograms to pounds.
Assistant -> tool_call: convert_units({"value": 42, "from_unit": "km", "to_unit": "mi"})
Assistant -> tool_call: convert_units({"value": 5, "from_unit": "kg", "to_unit": "lb"})
Tool [convert_units] -> 26.0976 mi
Tool [convert_units] -> 11.0231 lb
Assistant: 42 kilometers is approximately 26.0976 miles, and 5 kilograms is approximately 11.0231 pounds.

tool calls this run: 2 across 1 turn(s)


Two `convert_units` calls, one turn, two tool-result messages, one grounded answer. The model recognised the two conversions are independent and issued them together.

## Step 12 — Structured vs text: the reliability head-to-head

Now the honest measurement behind this whole chapter. `compare_structured_vs_text` asks the model the **same** real queries two ways — the structured function-calling path (JSON, schema-validated) vs the ReAct-style text path from Step 1 (`TOOL: name(args)`, regex-parsed) — and scores how often each yields a **parseable, dispatchable** call. Every generation is greedy, so this reproduces exactly.

In [13]:
from function_calling_agent import compare_structured_vs_text

rows = compare_structured_vs_text(model)
print(f"{'structured':>10} {'text':>6} | query")
print('-' * 64)
for r in rows:
    print(f"{'OK' if r.structured_ok else 'FAIL':>10} {'OK' if r.text_ok else 'FAIL':>6} "
          f'| {r.query[:44]}')

structured   text | query
----------------------------------------------------------------
        OK   FAIL | What is 481 multiplied by 32, then plus 19?
        OK   FAIL | What is 17 to the power of 3, minus 200?
        OK   FAIL | Convert 42 kilometres to miles.
        OK   FAIL | How many pounds is 5 kilograms?
        OK   FAIL | Convert 100 degrees Celsius to Fahrenheit.
        OK   FAIL | What is the exchange rate from US dollars to
        OK     OK | What is 1000 grams in pounds?
        OK   FAIL | What is 250 times 4, then divided by 5?


In [14]:
s_ok = sum(r.structured_ok for r in rows)
t_ok = sum(r.text_ok for r in rows)
print(f'structured (JSON, schema-validated): {s_ok}/{len(rows)} dispatchable '
      f'({s_ok / len(rows):.0%})')
print(f'text (TOOL: prose, regex-parsed)   : {t_ok}/{len(rows)} dispatchable '
      f'({t_ok / len(rows):.0%})')

structured (JSON, schema-validated): 8/8 dispatchable (100%)
text (TOOL: prose, regex-parsed)   : 1/8 dispatchable (12%)


On this real set, structured calling yields a dispatchable call almost every time, while the text-parsing path fails on most queries — the model simply does not reliably format prose into the rigid `TOOL:` grammar the regex needs. **That gap is the value of structured function calling**, measured on real output. It is the same brittleness the ReAct chapter's parser fights, quantified.

## Step 13 — The figures on the chapter page come from exactly this run

Every figure in the chapter is generated from the same real module you just ran — no hand-typed numbers. The single-call and parallel trace figures are real solved traces (Steps 9 and 11); the reliability bars are the real comparison (Step 12). You can regenerate them yourself:

```bash
python "../../tools/make_figures_03.py"   # writes agentic03_*.png into ../../images/
```

That closes the loop between the page, the notebook, and the module: one real agent, demonstrated three ways, always in agreement.

In [15]:
# Confirm the numbers behind the chapter's reliability figure, from THIS run:
print('structured dispatchable :', sum(r.structured_ok for r in rows), '/', len(rows))
print('text dispatchable       :', sum(r.text_ok for r in rows), '/', len(rows))
print('parallel-trace tool calls:', par.num_tool_calls, 'in', par.num_turns, 'turn(s)')

structured dispatchable : 8 / 8
text dispatchable       : 1 / 8
parallel-trace tool calls: 2 in 1 turn(s)


## Recap — what you built

You built a **real function-calling agent** end to end: three real JSON-schema tools, the schema-into-prompt mechanism, structured `<tool_call>` parsing, schema validation, the call → tool-result → answer protocol loop with real message roles, single / sequential / **parallel** multi-tool traces, and an honest structured-vs-text reliability comparison — all driving a genuine LLM with reproducible greedy decoding.

The one idea to keep: **declaring a typed schema and parsing a structured call is more reliable than parsing free text.** ReAct proved a model can *act*; function calling makes that acting robust by moving the tool call from fuzzy prose into a data format the runtime can trust — and by validating the arguments before it ever runs a tool.

Next: [Model Context Protocol (MCP)](../../08-Model-Context-Protocol-MCP/08-Model-Context-Protocol-MCP.md) (the open standard that lets *any* client discover and call *any* server's tools over the same schema-and-structured-call idea), and back to [ReAct](../../02-ReAct-Reason-and-Act/02-ReAct-Reason-and-Act.md) (the reasoning loop these tool calls slot into).